# Method Comparison & Ensemble Ranking

Compares all 6 similarity metrics side-by-side and builds ensemble rankings.
Requires `metrics_results.pkl` from `analysis_extended.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os, pickle

%matplotlib inline

results = pickle.load(open('metrics_results.pkl', 'rb'))
constraint_patterns = results['constraint_patterns']
constraint_patternsLT = results['constraint_patternsLT']
all_methods = results['all_methods']
all_methodsLT = results['all_methodsLT']
method_names = list(all_methods.keys())

def get_top_n(scores, cluster, pattern, n=20, ascending=True):
    s = scores[cluster][pattern]
    return set(s.nsmallest(n).index) if ascending else set(s.nlargest(n).index)

names_str = ', '.join(method_names)
print(f'Loaded {len(method_names)} methods: {names_str}')
print('Clusters (raw):', list(constraint_patterns.keys()))
print('Clusters (LT): ', list(constraint_patternsLT.keys()))

In [ ]:
def plot_genes_per_method(method_name, method_scores, patterns, lt=False, n=20, ascending=True):
    """Plot top-N gene expression curves overlaid with the constraint pattern."""
    x = [0, 3, 6, 9]
    suffix = "LT" if lt else ""
    for cluster_name, pattern_dict in patterns.items():
        gene_file = f"analysis data/gene_counts{suffix}/{cluster_name}_annotated.csv"
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x
        for pattern_name, pattern_vals in pattern_dict.items():
            scores = method_scores[cluster_name][pattern_name]
            top_genes = scores.nsmallest(n).index.tolist() if ascending else scores.nlargest(n).index.tolist()
            fig, ax = plt.subplots(figsize=(10, 6))
            plotted = 0
            for gene in top_genes:
                if gene in gene_df.index:
                    ax.plot(x, gene_df.loc[gene], color="steelblue", alpha=0.4, linewidth=1)
                    plotted += 1
            ax.plot(x, pattern_vals, color="crimson", linewidth=2.5, linestyle="--")
            lt_tag = " (LT)" if lt else ""
            title = method_name + ' | ' + cluster_name + ' | Top ' + str(n) + ' genes' + lt_tag
            ax.set_title(title)
            ax.set_xlabel("Timepoint")
            ax.set_ylabel("Gene counts")
            ax.set_xticks(x)
            legend_elements = [
                Line2D([0], [0], color="crimson", linewidth=2.5, linestyle="--", label="Constraint pattern"),
                Line2D([0], [0], color="steelblue", alpha=0.4, label="Top " + str(n) + " genes (n=" + str(plotted) + ")"),
            ]
            ax.legend(handles=legend_elements, loc="upper right")
            plt.tight_layout()
            plt.show()

## Per-method plots (all 6 methods)

Same style as the original notebook — top 20 gene expression curves overlaid with the constraint pattern.

In [ ]:
# Raw data plots for all 6 methods
for name, (scores, asc) in all_methods.items():
    sep = "=" * 40
    print(sep, name, sep)
    plot_genes_per_method(name, scores, constraint_patterns, lt=False, ascending=asc)

In [ ]:
# Log-transformed data plots for all 6 methods
for name, (scores, asc) in all_methodsLT.items():
    sep = "=" * 40
    print(sep, name, '(LT)', sep)
    plot_genes_per_method(name, scores, constraint_patternsLT, lt=True, ascending=asc)

## Phase 6: Ensemble Ranking

Converting all metric scores to ranks (1 = best), then aggregating.

In [ ]:
def compute_ranks(methods_dict, cluster_name, pattern_name):
    """Convert raw scores to ranks (1 = best) for each method.
    Returns a DataFrame with genes as index, methods as columns."""
    rank_df = pd.DataFrame()
    for mname, (scores, ascending) in methods_dict.items():
        s = scores[cluster_name][pattern_name]
        # ascending=True means lower is better, so rank ascending
        # ascending=False means higher is better, so rank descending
        rank_df[mname] = s.rank(ascending=ascending, method='min')
    return rank_df

# Demo: show rank distribution for clusterOne
demo_ranks = compute_ranks(all_methods, "clusterOne", "constraint")
print("Rank DataFrame shape:", demo_ranks.shape)
print("\nTop 10 genes by Pearson rank:")
print(demo_ranks.sort_values("Pearson").head(10))

### Vote count: how many methods place each gene in the top 20?

In [ ]:
def ensemble_vote_count(methods_dict, cluster_name, pattern_name, n=20):
    """Count how many methods place each gene in top N."""
    rank_df = compute_ranks(methods_dict, cluster_name, pattern_name)
    votes = (rank_df <= n).sum(axis=1)
    avg_rank = rank_df.mean(axis=1)
    result = pd.DataFrame({"votes": votes, "avg_rank": avg_rank})
    result = result.sort_values(["votes", "avg_rank"], ascending=[False, True])
    return result

print("Ensemble Vote Count (Top 20 threshold, raw data)")
print("=" * 60)
nm = len(method_names)
for cluster_name in constraint_patterns:
    votes = ensemble_vote_count(all_methods, cluster_name, "constraint", n=20)
    top20 = votes.head(20)
    print()
    print('---', cluster_name, '---')
    print('Max votes possible:', nm)
    full_votes = (top20["votes"] == nm).sum()
    near_votes = (top20["votes"] >= nm - 1).sum()
    print('Genes with', str(nm) + '/' + str(nm), 'votes:', full_votes)
    print('Genes with >=' + str(nm-1) + '/' + str(nm), 'votes:', near_votes)
    print(top20.to_string())

### Average/median rank aggregation

In [ ]:
def ensemble_avg_rank(methods_dict, cluster_name, pattern_name, n=20):
    """Aggregate by average and median rank across all methods."""
    rank_df = compute_ranks(methods_dict, cluster_name, pattern_name)
    result = pd.DataFrame({
        "mean_rank": rank_df.mean(axis=1),
        "median_rank": rank_df.median(axis=1),
        "min_rank": rank_df.min(axis=1),
        "max_rank": rank_df.max(axis=1),
    })
    return result.sort_values("mean_rank").head(n)

print("Ensemble Average Rank (Top 20, raw data)")
print("=" * 60)
for cluster_name in constraint_patterns:
    avg = ensemble_avg_rank(all_methods, cluster_name, "constraint", n=20)
    print()
    print('---', cluster_name, '---')
    print(avg.to_string())

In [ ]:
# Same for LT data
print("Ensemble Average Rank (Top 20, LT data)")
print("=" * 60)
for cluster_name in constraint_patternsLT:
    avg = ensemble_avg_rank(all_methodsLT, cluster_name, "constraint", n=20)
    print()
    print('---', cluster_name, '---')
    print(avg.to_string())

### Ensemble plots

Opacity = consensus strength. Darker lines = more methods agree on that gene.

In [ ]:
def plot_ensemble_top_genes(methods_dict, pat_dict, lt=False, n=20):
    """Plot ensemble top-N genes. Opacity = consensus strength."""
    x = [0, 3, 6, 9]
    suffix = "LT" if lt else ""
    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f"analysis data/gene_counts{suffix}/{cluster_name}_annotated.csv"
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x
        for pattern_name, pattern_vals in pattern_dict.items():
            rank_df = compute_ranks(methods_dict, cluster_name, pattern_name)
            avg_ranks = rank_df.mean(axis=1).sort_values()
            top_genes = avg_ranks.head(n).index.tolist()
            votes = (rank_df.loc[top_genes] <= n).sum(axis=1)
            fig, ax = plt.subplots(figsize=(10, 6))
            max_votes = len(methods_dict)
            for gene in top_genes:
                if gene in gene_df.index:
                    v = votes[gene]
                    alpha = 0.2 + 0.6 * (v / max_votes)
                    ax.plot(x, gene_df.loc[gene], color="steelblue", alpha=alpha, linewidth=1)
            ax.plot(x, pattern_vals, color="crimson", linewidth=2.5, linestyle="--")
            lt_tag = " (LT)" if lt else ""
            title = 'Ensemble (avg rank) | ' + cluster_name + ' | Top ' + str(n) + ' genes' + lt_tag
            ax.set_title(title)
            ax.set_xlabel("Timepoint")
            ax.set_ylabel("Gene counts")
            ax.set_xticks(x)
            legend_elements = [
                Line2D([0], [0], color="crimson", linewidth=2.5, linestyle="--", label="Constraint pattern"),
                Line2D([0], [0], color="steelblue", alpha=0.8, label="High consensus (>=" + str(max_votes-1) + " votes)"),
                Line2D([0], [0], color="steelblue", alpha=0.3, label="Low consensus (1-2 votes)"),
            ]
            ax.legend(handles=legend_elements, loc="upper right")
            plt.tight_layout()
            plt.show()

print("Raw data:")
plot_ensemble_top_genes(all_methods, constraint_patterns, lt=False)
print()
print("Log-transformed data:")
plot_ensemble_top_genes(all_methodsLT, constraint_patternsLT, lt=True)

### Ensemble vs individual method overlap

In [ ]:
# Ensemble top-20 vs individual method top-20 overlap
print("Ensemble vs Individual Method Overlap (Top 20, raw data)")
print("=" * 60)

for cluster_name in constraint_patterns:
    rank_df = compute_ranks(all_methods, cluster_name, "constraint")
    ensemble_top = set(rank_df.mean(axis=1).nsmallest(20).index)
    print()
    print(cluster_name + ':')
    for mname, (scores, asc) in all_methods.items():
        method_top = get_top_n(scores, cluster_name, "constraint", n=20, ascending=asc)
        overlap = len(ensemble_top & method_top)
        print('  Ensemble vs ' + mname + ': ' + str(overlap) + '/20 shared')

### Final ensemble table

In [ ]:
print("FINAL ENSEMBLE RANKINGS")
print("=" * 80)

for label, methods_dict, pat_dict in [
    ("Raw", all_methods, constraint_patterns),
    ("LT", all_methodsLT, constraint_patternsLT)
]:
    print()
    print("=" * 80)
    print(label, 'DATA')
    print("=" * 80)
    for cluster_name in pat_dict:
        rank_df = compute_ranks(methods_dict, cluster_name, "constraint")
        avg_ranks = rank_df.mean(axis=1).sort_values()
        top20 = avg_ranks.head(20)
        votes = (rank_df.loc[top20.index] <= 20).sum(axis=1)
        print()
        print('---', cluster_name, '---')
        print('Rank  Gene                 Avg Rank   Votes  Methods in top-20')
        print('-' * 75)
        for i, (gene, avg_r) in enumerate(top20.items(), 1):
            v = votes[gene]
            in_top = [m for m in method_names if rank_df.loc[gene, m] <= 20]
            methods_str = ', '.join(in_top)
            print(f'{i:<6}{gene:<20}{avg_r:>10.1f}{v:>8}  {methods_str}')

## Phase 7: Code Review

### Metric verification — known test vectors

In [ ]:
# Test with known vectors
test_pattern = np.array([0.0, 1.0, 0.5, 0.5])
test_identical = np.array([0.0, 1.0, 0.5, 0.5])
test_opposite = np.array([1.0, 0.0, 0.5, 0.5])

def check(label, actual, expected, tol=1e-10):
    ok = abs(actual - expected) < tol
    status = 'OK' if ok else 'FAIL'
    print(f'  {label}: {actual:.4f} (expected: {expected}) {status}')

def check_gt(label, actual, threshold=0):
    ok = actual > threshold
    status = 'OK' if ok else 'FAIL'
    print(f'  {label}: {actual:.4f} (expected: >{threshold}) {status}')

print("=== Metric Verification ===")

from scipy import stats as scipy_stats
from scipy.spatial.distance import cosine as cosine_distance
from dtw import dtw as dtw_func
import similaritymeasures

r_ident, _ = scipy_stats.pearsonr(test_pattern, test_identical)
r_oppos, _ = scipy_stats.pearsonr(test_pattern, test_opposite)
print("Pearson:")
check('identical', r_ident, 1.0)
check('opposite', r_oppos, -1.0)

dtw_ident = dtw_func(test_pattern, test_identical).distance
dtw_oppos = dtw_func(test_pattern, test_opposite).distance
print("DTW:")
check('identical', dtw_ident, 0.0)
check_gt('opposite', dtw_oppos)

cos_ident = 1.0 - cosine_distance(test_pattern, test_identical)
print("Cosine:")
check('identical', cos_ident, 1.0)

x_pts = np.array([0, 3, 6, 9], dtype=float)
c1 = np.column_stack([x_pts, test_pattern])
c2 = np.column_stack([x_pts, test_identical])
fd = similaritymeasures.frechet_dist(c1, c2)
print("Frechet:")
check('identical', fd, 0.0)

mse_i = np.mean((test_pattern - test_identical) ** 2)
mse_o = np.mean((test_pattern - test_opposite) ** 2)
print("MSE:")
check('identical', mse_i, 0.0)
check('opposite', mse_o, 0.5)

eps = 1e-10
sm = np.mean(2.0 * np.abs(test_pattern - test_identical) /
             (np.abs(test_pattern) + np.abs(test_identical) + eps))
print("sMAPE:")
check('identical', sm, 0.0, tol=1e-6)

print()
print("=== All metrics verified ===")

### Edge case audit

In [ ]:
print("=== Edge Case Audit ===")
print()

def normalize_pattern(pv):
    eps = 1e-10
    p = np.array(pv, dtype=float)
    return (p - p.min() + eps) / (p.max() - p.min() + eps)

def load_gene_counts(lt=False):
    suffix = "LT" if lt else ""
    folder = f'analysis data/gene_counts{suffix}/'
    cc = {}
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith('.csv'): continue
        df = pd.read_csv(os.path.join(folder, fn), index_col=0)
        df.columns = [0, 3, 6, 9]
        cc[fn.split('_')[0]] = df
    return cc

def normalize_01(cluster_counts):
    eps = 1e-10
    out = {}
    for cn, df in cluster_counts.items():
        rmin, rmax = df.min(axis=1), df.max(axis=1)
        rr = rmax - rmin
        ndf = df.sub(rmin, axis=0).add(eps).div(rr.add(eps), axis=0)
        ndf.loc[rr == 0] = 1.0
        out[cn] = ndf
    return out

gene_counts_norm = normalize_01(load_gene_counts(False))
constraints = {
    "clusterOne": [1.45, 12.0, 5.21, 6.51],
    "clusterTwo": [0.371, 0.803, 1.22, 1.78],
    "clusterThree": [0.0405, 0, 0.0369, 0.0353],
    "clusterFour": [0.00913, 0.638, 0.0178, 0.0243]
}

print("1. Constant genes (zero variance across timepoints):")
for name, df in gene_counts_norm.items():
    n_constant = (df.std(axis=1) == 0).sum()
    n_all_ones = (df == 1.0).all(axis=1).sum()
    print(f'  {name}: {n_constant} constant -> all 1.0 ({n_all_ones} confirmed)')

print()
print("2. Zero values in constraint patterns:")
for name, vals in constraints.items():
    zeros = [i for i, v in enumerate(vals) if v == 0]
    if zeros:
        normed = normalize_pattern(vals)
        print(f'  {name}: zero at index {zeros} -> normalized: {normed}')
    else:
        print(f'  {name}: no zeros')

print()
print("3. NaN/Inf check in metric results:")
has_issues = False
for mname, (scores, _) in all_methods.items():
    for cluster_name in scores:
        s = scores[cluster_name]["constraint"]
        n_nan = s.isna().sum()
        n_inf = np.isinf(s).sum()
        if n_nan > 0 or n_inf > 0:
            print(f'  WARNING: {mname}/{cluster_name}: {n_nan} NaN, {n_inf} Inf')
            has_issues = True
if not has_issues:
    print("  No NaN/Inf values found.")

print()
print("4. DTW normalization consistency:")
print("   Pattern and genes both use (x - min + eps) / (max - min + eps) -> consistent.")

print()
print("5. clusterFour spike analysis:")
c4 = np.array(constraints["clusterFour"])
print(f'   Raw: {c4}')
print(f'   Spike ratio t=3/t=0: {c4[1]/c4[0]:.1f}x')
normed = normalize_pattern(c4)
print(f'   Normalized: {normed}')
print("   -> Effectively [0, 1, 0, 0]: a pure spike. All methods struggle.")

print()
print("=== Edge case audit complete ===")

---
## Results

Full results, overlap matrices, theoretical analysis, and recommendations
are in **`SUMMARY_REPORT.md`**.

**Quick takeaways:**
- Pearson + Frechet + MSE form a tight consensus for most clusters
- Cosine conflates magnitude with shape (no mean-centering) — not ideal here
- sMAPE fails when the pattern passes through zero (clusterThree: 0/20 overlap with all others)
- clusterFour (70x spike) defeats all methods — needs denser time sampling
- Recommended ensemble: Pearson + DTW + Frechet + MSE, require >= 3/4 votes